This notebook is used to generate all the necessary files to run the dashboard smoothly. 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from rapidfuzz import process
import pickle
import numpy as np
import matplotlib.pyplot as plt
import os
import pickle

Generate a file containing the name of each aviary, the species within the aviary, the number and gender of the individuals and the name of the related metadata file


In [5]:
metadata_path = 'metadata_aviaries/'
species_df = pd.read_csv(metadata_path + 'Avieries_obsolete.csv', encoding='latin-1')

number_species = species_df.groupby('Aviary')['Common name'].nunique()
number_individuals = species_df.groupby('Aviary')['Total count'].sum()
species_list = species_df.groupby('Aviary')['Common name'].apply(list)
individuals_list = species_df.groupby('Aviary')['Total count'].apply(list)
individuals_gender_list = species_df.groupby('Aviary')['Genders (male.female.unknown)'].apply(list)
metadata_paths = species_df.groupby('Aviary')['preprocessed data'].first()  

combined_df = pd.DataFrame({'species': species_list, 'individuals': individuals_list, 'individuals_genders (m.f.u)': individuals_gender_list, 'Number of Species': number_species, 'Number of Individuals': number_individuals, "preprocessed data": metadata_paths})
combined_df = combined_df.dropna() # Some aviaries don't have metadata files, so we drop those rows for now

# Link preprocessed data path to metadata path as they are not in the same format
aviaries_metadata = [f for f in os.listdir(metadata_path) if f.endswith('.xlsx')]
for idx, row in combined_df.iterrows():
    avirary_path = row["preprocessed data"]
    best_match = process.extractOne(avirary_path, aviaries_metadata)
    if best_match and best_match[1] > 80:
        file_path = best_match[0]

    combined_df.loc[row.name, "Metadata_filename"] = file_path

combined_df.reset_index(inplace=True)

combined_df.to_csv("general_aviary_data.csv", index=False)


Next we pre process all the metadata files so that the dsahboard is able to update quickly 

In [2]:
def process_metadata(df):
    columns_to_drop = ["Unnamed: 0.1", "Unnamed: 0", "sessionId", "time", "th1", "th1_value", 'th2', 'th2_value', 'th3', 'th3_value', 'wudate', 'lon', 'lat']
    to_drop = [s for s in df.columns if s in columns_to_drop]
    print(to_drop)
    df.drop(columns=to_drop, inplace=True)
    df["fusion_model_prediction"] = df["fusion_model_prediction"].replace("NO PREDICTION (0.0000)", None)
    df["Call_Presence"] = df["fusion_model_prediction"].notnull().astype(int)

    df["Final prediction"] = None
    
    for idx, row in df.iterrows():
        lines = str(row["fusion_model_prediction"]).split("\n")
        for line in lines:
            subset = line.split(" ")
            try:
                if subset[0] == "nan":
                    row["Final prediction"] = None
                    continue

                tmp_pred = ""
                for string in subset[:-1]:
                    tmp_pred += string + " "

                if df.loc[idx, "Final prediction"] is not None:
                    df.loc[idx, "Final prediction"] += ", " + tmp_pred
                else:
                    df.loc[idx, "Final prediction"] = tmp_pred

            except Exception as error:
                print(f"Error processing row {idx}: {error}")

    return df
    
# Filter to keep only relevant MIT AST classes and create a new dataframe storing all the information required for the visualisations
def filter_metadata(df, native_species):
    MIT_classes_of_interest = ["Crowd", "Civil defense siren", "Railroad car, train wagon", "Vehicle", "Motorcycle", "Thunderstorm", "Air horn, truck horn", "Engine starting", "Siren", "Medium engine (mid frequency)", "Thunder", "Train", "Car", "Vehicle horn, car horn, honking", "Roaring cats (lions, tigers)", "Roar", "Dog"]
    ROAR_VARIANTS = ["Roar", "Roaring cats (lions, tigers)", "Roaring cat"]

    # Events
    if "MIT_AST_label" in df.columns:
        df["event"] = df["MIT_AST_label"].where(df["MIT_AST_label"].isin(MIT_classes_of_interest), None)
        df["event"] = df["event"].replace(ROAR_VARIANTS, "Roar")
    else:
        df["event"] = None

    # Explode predictions into one row per species
    df["Final prediction"] = df["Final prediction"].fillna("")
    df_exploded = df[df["Final prediction"] != ""].copy()
    df_exploded["species_list"] = df_exploded["Final prediction"].str.split(", ")
    df_exploded = df_exploded.explode("species_list")
    df_exploded["species_list"] = df_exploded["species_list"].str.strip().str.lower()

    # Match native species
    rows = []
    for _, row in df_exploded.iterrows():
        if "predicted_call_type_birdnet" in df_exploded.columns:
            call_type = row["predicted_call_type_birdnet"]
        else: 
            call_type = None

        sp = row["species_list"]
        matched = next((n for n in native_species if n.lower() in sp or sp in n.lower()), None)
        rows.append({
            "datetime": row["datetime"],
            "species": matched,
            "wild specie": None if matched else sp,
            "call_type": call_type,
            "event": row["event"]
        })

    plot_df = pd.DataFrame(rows)
    return plot_df


def format_data(species):
        return species.replace("'", "").replace("[", "").replace("]", "").replace('"', "").strip().lower()

In [3]:
general_df = pd.read_csv("general_aviary_data.csv")
FOLDER_PATH = "metadata_aviaries/"
DESTINATION_FOLDER = "processed_data/"

if not os.path.exists(DESTINATION_FOLDER):
    os.makedirs("processed_data")

print(general_df["Metadata_filename"].tolist())
# Pre process the metadata so dashboard does not have to do it over and over, saves a lot of time
for file in os.listdir(FOLDER_PATH):
    print(file)
    print(file in general_df["Metadata_filename"].tolist())
    if file.endswith(".xlsx") and file in general_df["Metadata_filename"].tolist():
        general_row = general_df.loc[general_df["Metadata_filename"]==file]
        native_species = general_row["species"].iloc[0].split(",")
        native_species = [format_data(s.strip()) for s in native_species]

        aviary_name = general_row["Aviary"].tolist()[0]
        destination_path = DESTINATION_FOLDER + aviary_name + "_processed.csv"

        if os.path.exists(destination_path):
            continue

        df = pd.read_excel(FOLDER_PATH + file)
        processed = process_metadata(df)
        filtered = filter_metadata(processed, native_species)

        filtered.to_csv(destination_path, index=False)



['fl_avifauna_flamingos_sept25_data_meta.xlsx', 'fl_avifauna_vultures_sept25_data_meta.xlsx', nan, 'fl_beekse_bergen_20250404_meta.xlsx', 'fl_beekse_bergen_20250404_meta.xlsx', 'fl_beekse_bergen_20250412_meta.xlsx', nan, 'fl_blijdorp_flamingos_dec2025_data_meta.xlsx', 'fl_cologne_zoo_flamingos_nov2025_data_meta.xlsx', nan, 'fl_gaia_zoo_congo_15aug25_data_meta.xlsx', 'fl_gaia_zoo_savannah_08aug25_data_meta.xlsx', 'fl_gaia_zoo_taiga_18Jul25_data_meta.xlsx', 'fl_zoo_eindhoven_20250308_meta.xlsx', 'fl_zoo_eindhoven_20250503_meta.xlsx', 'fl_zoo_eindhoven_20250315_meta.xlsx', 'fl_zoo_helsinki_20250624_meta.xlsx', 'fl_zoo_helsinki_20250701_meta.xlsx', 'fl_zoo_parc_aug25_data_meta.xlsx', 'fl_zoo_eindhoven_20250426_meta.xlsx']
Avieries_obsolete.csv
False
fl_avifauna_flamingos_sept25_data_meta.xlsx
True
['Unnamed: 0.1', 'Unnamed: 0', 'sessionId', 'time', 'th1', 'th1_value', 'th2', 'th2_value', 'th3', 'th3_value', 'wudate', 'lon', 'lat']
fl_avifauna_storage_ibises_sept25_data_meta.xlsx
False
fl_a